# Data analysis

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib

import warnings

warnings.simplefilter("ignore")

In [ ]:
DATA_PATH = pathlib.Path.cwd() / "data" / "elucidata_ai_challenge_data.h5"

In [ ]:
def load_data(file_path):
    with h5py.File(file_path, "r") as h5file:
        train_images = {k: np.array(v) for k, v in h5file["images/Train"].items()}
        train_spots = {k: np.array(v) for k, v in h5file["spots/Train"].items()}
        test_images = {k: np.array(v) for k, v in h5file["images/Test"].items()}
        test_spots = {k: np.array(v) for k, v in h5file["spots/Test"].items()}
    return train_images, train_spots, test_images, test_spots

In [ ]:
train_images, train_spots, test_images, test_spots = load_data(DATA_PATH)

In [ ]:
print("Dataset Overview:")
print(f"  Number of training slides: {len(train_images)}")
print(f"  Number of test slides: {len(test_images)}")
print("\nTraining Images Structure:")
for slide_name, image in train_images.items():
    print(f"  Slide: {slide_name}, Shape: {image.shape}, Data Type: {image.dtype}")
print("\nTraining Spots Structure:")
for spot_name, spot in train_spots.items():
    print(f"  Spot: {spot_name}, Shape: {spot.shape}, Data Type: {spot.dtype}")
print("Spot values")
print(train_spots["S_1"][0])

print("\nTest Images Structure:")
for slide_name, image in test_images.items():
    print(f"  Slide: {slide_name}, Shape: {image.shape}, Data Type: {image.dtype}")
print("\Test Spots Structure:")
for spot_name, spot in test_spots.items():
    print(f"  Spot: {spot_name}, Shape: {spot.shape}, Data Type: {spot.dtype}")

In [ ]:
sns.set_style("whitegrid")

num_slides = len(train_spots)
cols = 3
rows = (num_slides // cols) + (num_slides % cols > 0)

fig, axes = plt.subplots(rows, cols, figsize=(8, 3 * rows), dpi=100)

axes = axes.flatten() if num_slides > 1 else [axes]

# Aesthetic color palette
spot_color = "#000000"  # black color for spots

for i, (slide_name, spots) in enumerate(train_spots.items()):
    axes[i].scatter(
        spots["x"],
        spots["y"],
        s=5,
        color=spot_color,
        alpha=1,
        edgecolors="w",
        linewidth=0.5,
    )
    axes[i].set_title(
        f"Spot Distribution - {slide_name}",
        fontsize=14,
        fontweight="bold",
        color="#ff0808",
    )
    axes[i].set_xlabel("X Coordinate", fontsize=12)
    axes[i].set_ylabel("Y Coordinate", fontsize=12)
    axes[i].invert_yaxis()
    axes[i].tick_params(axis="both", labelsize=10)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 3))
for i, slide_name in enumerate(list(train_images.keys())[:3]):
    plt.subplot(1, 3, i + 1)
    plt.imshow(train_images[slide_name])
    plt.title(f"Train - {slide_name}")
    plt.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(3, 3))
# Display the test image
plt.imshow(test_images["S_7"])
plt.title("Test - S_7")
plt.axis("off")
plt.show()

## Cell types

In [ ]:
all_train_spots_df = pd.DataFrame(
    [xs for x in train_spots.values() for xs in x.tolist()],
    columns=["x", "y"] + [f"C{idx}" for idx in range(1, 36)],
)


cell_type_cols = [col for col in all_train_spots_df.columns if col.startswith("C")]
cell_type_stats = all_train_spots_df[cell_type_cols].describe()

print("Descriptive Statistics for Cell Type Abundances:")
print(cell_type_stats[cell_type_cols[:10]])
print(cell_type_stats[cell_type_cols[10:20]])
print(cell_type_stats[cell_type_cols[20:]])

In [ ]:
sns.set_theme(rc={"figure.figsize": (20, 6)})
sns.boxplot(all_train_spots_df[cell_type_cols], showfliers=False)

In [ ]:
cell_type_corr = all_train_spots_df[cell_type_cols].corr()

plt.figure(figsize=(10, 6))
sns.heatmap(cell_type_corr, annot=False, cmap="Blues")
plt.title("Correlation Matrix of Cell Type Abundances")
plt.show()